**TRAINING A CUSTOM MODEL**

In [5]:
import sys
print(sys.executable)

c:\Users\joshu\AppData\Local\Programs\Python\Python312\python.exe


In [6]:
import sys

!{sys.executable} -m pip uninstall -y transformers tokenizers datasets evaluate accelerate huggingface-hub safetensors
!{sys.executable} -m pip install --no-cache-dir "transformers==4.46.3" "tokenizers>=0.20,<0.21" "datasets" "evaluate" "accelerate" "safetensors"

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.21.4
Uninstalling tokenizers-0.21.4:
  Successfully uninstalled tokenizers-0.21.4
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: safetensors 0.4.4
Uninstalling safetensors-0.4.4:
  Successfully uninstalled safetensors-0.4.4


   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 1.2 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/10.0 MB 1.1 MB/s eta 0:00:09
   ----- ---------------------------------- 1.3/10.0 MB 1.5 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/10.0 MB 1.7 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/10.0 MB 2.0 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/10.0 MB 2.1 MB/s eta 0:00:04
   --------------- ------------------------ 3.9/10.0 MB 2.4 MB/s eta 0:00:03
   ------------------- -------------------- 5.0/10.0 MB 2.6 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/10.0 MB 2.9 MB/s eta 0:00:02
   ------------------------------ --------- 7.6/10.0 MB 3.3 MB/s eta 0:00:01
   -----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference 0.51.2 requires aiortc~=1.9.0, but you have aiortc 1.14.0 which is incompatible.
inference 0.51.2 requires click<8.2.0, but you have click 8.4.1 which is incompatible.
inference 0.51.2 requires filelock<=3.17.0,>=3.12.0, but you have filelock 3.29.4 which is incompatible.
inference 0.51.2 requires numpy<2.3.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
inference 0.51.2 requires packaging~=24.0, but you have packaging 26.2 which is incompatible.
inference 0.51.2 requires typer<=0.12.5,>=0.9.0, but you have typer 0.25.1 which is incompatible.
inference 0.51.2 requires typing_extensions<=4.12.2,>=4.8.0, but you have typing-extensions 4.15.0 which is incompatible.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoConfig
import pandas as pd 
import numpy as np


training_set = pd.read_csv("notebooks/train.csv")
test_set = pd.read_csv("notebooks/test.csv")
validation_set = pd.read_csv("notebooks/validation.csv")

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")  #model weights














FileNotFoundError: [Errno 2] No such file or directory: 'notebooks\\test.csv'

In [ ]:
#tokenizaiton 

def tokenize_function(training):
    return tokenizer(training["text"], padding="max_length", truncation=True)


training_tokens = training_set.map(tokenize_function, batched=True)
test_tokens = test_set.map(tokenize_function, batched=True)
validation_tokens = validation_set.map(tokenize_function, batched=True)


In [ ]:
training_args = TrainingArguments(
    output_dir="notebooks\\results",  # output directory
    num_train_epochs=3,  # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=64,  # batch size for evaluation
    warmup_steps=500,  # number of warmup steps for learning rate scheduler
    weight_decay=0.01,  # strength of weight decay
    logging_dir="notebooks\\logs",  # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,  
    args=training_args,  
    train_dataset=training_set,  # training dataset
    eval_dataset=test_set,  # evaluation dataset
)